# Week 2: Mini Semantic Search
Load FAQs with Pandas, clean them, chunk the text, embed it into Chroma, and print the top 3 matches for a query.

In [ ]:
!pip install -q chromadb pandas numpy

In [ ]:
%%writefile faqs.csv
question,answer
How do I create an account?,Click Sign Up and fill in your name email and password.
How do I change my password?,Go to Settings and select Change Password.
What should I do if I forget my password?,Click Forgot Password on the login page and follow the instructions sent to your email.
How can I update my email address?,Open your account settings and edit your email address.
How do I update my profile information?,Go to your profile page and click Edit Profile.
How do I delete my account?,Open Settings and select Delete Account if the option is available.
How do I log out?,Click your profile icon and select Log Out.
Why can't I log in?,Check your email and password and make sure your internet connection is working.
How do I contact customer support?,Email support@example.com or use the live chat feature.
What are your support hours?,Our support team is available Monday through Friday during business hours.
How long does it take to receive a response from support?,We typically respond within one business day.
How can I track my order?,Open your Orders page and select the order you want to track.
How do I cancel an order?,Visit your Orders page and select Cancel Order if cancellation is still available.
Can I change my shipping address?,Contact support as soon as possible if your order has not shipped.
How long does delivery take?,Delivery usually takes three to seven business days depending on your location.
What payment methods do you accept?,We accept major credit cards and PayPal.
Why was my payment declined?,Check your payment details and available balance or contact your bank.
How can I get a refund?,Contact customer support with your order number and refund request.
How long do refunds take?,Refund processing times depend on your payment provider and bank.
Where can I find my receipt?,Open your order history and select the order to view its receipt.
How do I cancel my subscription?,Go to Settings > Billing and click Cancel Subscription.
How can I upgrade my subscription?,Open your Billing page and select an available upgrade plan.
How do I downgrade my subscription?,Go to Billing and choose a lower subscription plan.
When will my subscription renew?,Check your Billing page for your next renewal date.
What happens when my subscription expires?,You may lose access to paid features until you renew your subscription.
Can I change my subscription payment method?,Open Billing and update your saved payment method.
Is there a free plan?,Check the Pricing page for available free plans and their limitations.
How do I reset my account settings?,Open Settings and select the preferences you want to change.
How do I enable notifications?,Go to Settings > Notifications and turn on your preferred alerts.

## 1. Load the raw dataset

In [ ]:
import re
import numpy as np
import pandas as pd
import chromadb

def load_data(path="faqs.csv"):
    return pd.read_csv(path)

raw = load_data()
print("Raw shape:", raw.shape)
raw.head()

## 2. Clean the data with functions (text, missing values, duplicates, types, outliers)

In [ ]:
def clean_text(text):
    t = str(text).lower().strip()
    t = re.sub(r"[^a-z0-9@.\s]", "", t)
    return re.sub(r"\s+", " ", t)

def handle_missing(df):
    df = df.copy().replace(r"^\s*$", np.nan, regex=True)
    return df.dropna(subset=["question", "answer"])

def remove_duplicates(df):
    key = df["question"].map(clean_text)
    return df.loc[~key.duplicated()]

def fix_types(df):
    df = df.copy()
    df["question"] = df["question"].astype(str)
    df["answer"] = df["answer"].astype(str)
    return df

def remove_outliers(df, z_max=3):
    df = df.copy()
    df["n_words"] = df["answer"].str.split().str.len()
    counts = df["n_words"].to_numpy()
    z = (counts - np.mean(counts)) / (np.std(counts) or 1)
    return df.loc[np.abs(z) <= z_max]

def clean_data(df):
    df = handle_missing(df)
    df = remove_duplicates(df)
    df = fix_types(df)
    df = remove_outliers(df)
    df["text"] = (df["question"] + " " + df["answer"]).map(clean_text)
    return df.reset_index(drop=True)

df = clean_data(raw)
print("Clean shape:", df.shape)
df.head()

## 3. NumPy statistics on answer length

In [ ]:
def length_stats(df):
    n = df["answer"].str.split().str.len().to_numpy()
    return {
        "mean": float(np.mean(n)),
        "median": float(np.median(n)),
        "std": float(np.std(n)),
        "p95": float(np.percentile(n, 95)),
    }

length_stats(df)

## 4. Chunk the text (size=200 words, overlap=40). FAQ answers are short, so each becomes one chunk.

In [ ]:
def chunk(words, size=200, overlap=40):
    step = size - overlap
    return [words[i:i + size] for i in range(0, len(words), step)]

def build_chunks(df):
    chunks, ids, metas = [], [], []
    for i, row in df.iterrows():
        for j, piece in enumerate(chunk(row["text"].split())):
            chunks.append(" ".join(piece))
            ids.append(f"faq{i}_c{j}")
            metas.append({"question": row["question"], "answer": row["answer"]})
    return chunks, ids, metas

chunks, ids, metas = build_chunks(df)
print(len(chunks), "chunks")

## 5. Embed and store in Chroma (default model: all-MiniLM-L6-v2, 384 dimensions)

In [ ]:
db = chromadb.Client()
col = db.create_collection("faqs")
col.add(documents=chunks, ids=ids, metadatas=metas)
print("Stored:", col.count())

## 6. Query and print the top 3 matches

In [ ]:
def search(col, query, k=3):
    hits = col.query(query_texts=[query], n_results=k)
    print(f"\nQuery: {query}")
    for rank, (meta, dist) in enumerate(
            zip(hits["metadatas"][0], hits["distances"][0]), 1):
        print(f"  {rank}. {meta['question']}  (distance {dist:.3f})")
        print(f"     -> {meta['answer']}")

for q in ["I can't sign in",
          "I want my money back",
          "when does my plan renew",
          "how do I talk to a human"]:
    search(col, q)